In [ ]:
import numpy as np
import numpy.typing as npt
from astropy.time import Time
from sorts.propagator import SGP4
from sorts.space_object import SpaceObject
from sorts.radar.radars import get_radar
from sorts.types import Datetime64_us, Timedelta64_us, Float64_as_sec
from sorts.controller_v2.tracker_controller import TrackerController
from sorts.controller_v2.fence_scan_controller_new import FenceScanController
from sorts.schedule_v2 import Schedule, ExperimentDetail
from sorts.scheduler_v2.priority_scheduling import _priority_scheduling_df

# import for plottings
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
pd.set_option("display.expand_frame_repr", False)

In [3]:
epoch = Time(53005.0, format="mjd", scale="utc")  # 2004-01-01 00:00:00Z
# start_time = Time("2025-06-30 00:00:00")
# end_time = Time("2025-06-30 00:00:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms
start_time = Time("2025-01-01 00:00:00")
end_time = Time("2025-01-01 04:00:00")
control_slice_duration = np.timedelta64(int(60*1e6), "us")  # 10ms

eiscat3d = get_radar("eiscat3d", "stage1-array")

spobj = SpaceObject(
    SGP4,
    propagator_options={"settings": {"out_frame": "ITRF"}},
    a=7200e3,
    e=0.02,
    i=75,
    raan=86,
    aop=0,
    mu0=60,
    epoch=epoch,
    parameters={"d": 0.1},
)


exp_detail_1 = ExperimentDetail(
    id=1,
    coh_int_bandwidth=1.0,  # TODO: invtg: not used in `sorts.signals.hard_target_snr`?
    ipp=1.0,  # TODO: invtg: not used in `sorts.signals.hard_target_snr`?
    pulse_length=1.0,  # TODO: invtg: not used in `sorts.signals.hard_target_snr`?
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,  # TODO: invtg: not used in `sorts.signals.hard_target_snr`?
    noise_temp=150.0,
    slice_duration=np.timedelta64(10_000, "us"),  # 10ms
)

time_arr: npt.NDArray[Datetime64_us] = np.arange(
    start_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    end_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    control_slice_duration,
)
dt_arr: npt.NDArray[Timedelta64_us] = time_arr - epoch.to_value("datetime64").astype("datetime64[us]")  # type: ignore
dsec_arr: npt.NDArray[Float64_as_sec] = dt_arr.astype(np.float64) / 1e6  # type: ignore

ecefs = spobj.get_state(dsec_arr)

trackerController = TrackerController(
    tx_station=eiscat3d.tx[0],
    rx_stations=[],
    time=time_arr,
    space_object_states=ecefs,
    exp_num=0,
    # azimuth_range=None,
    # elevation_range=None,
)

fenceScanController = FenceScanController(
    tx_station=eiscat3d.tx[0],
    rx_station=[],
    exp_datail=exp_detail_1,
    azimuth=90, # sweep from east to west
    min_elevation=30,
    pointings_per_cycle=40,
)

In [4]:
tracker_schs = trackerController.generate()
tracker_tx_sch_df = tracker_schs.tx_schedule.as_dataframe()
tracker_tx_sch_df

,exp_num,pointing_az,pointing_el
stt_tstmp_us,,,
2025-01-01 00:00:00,0,-0.153980,-0.566812
2025-01-01 00:01:00,0,-0.120800,-0.611894
2025-01-01 00:02:00,0,-0.087521,-0.654860
2025-01-01 00:03:00,0,-0.054273,-0.695517
2025-01-01 00:04:00,0,-0.021183,-0.733684
...,...,...,...
2025-01-01 03:55:00,0,-0.258695,-0.544972
2025-01-01 03:56:00,0,-0.232776,-0.501636
2025-01-01 03:57:00,0,-0.205759,-0.456660


In [5]:
fence_schs = fenceScanController.generate(start_time, end_time)
fence_tx_sch_df = fence_schs.tx_schedule.as_dataframe()
fence_tx_sch_df

,exp_num,pointing_az,pointing_el
stt_tstmp_us,,,
2025-01-01 00:00:00.000,1,90.0,30.000000
2025-01-01 00:00:00.010,1,90.0,33.076923
2025-01-01 00:00:00.020,1,90.0,36.153846
2025-01-01 00:00:00.030,1,90.0,39.230769
2025-01-01 00:00:00.040,1,90.0,42.307692
...,...,...,...
2025-01-01 03:59:59.950,1,270.0,42.307692
2025-01-01 03:59:59.960,1,270.0,39.230769
2025-01-01 03:59:59.970,1,270.0,36.153846


In [6]:
# check schedule df memory usage
fence_tx_sch_df.memory_usage().sum()/1e6

np.float64(46.08)

In [ ]:
master_sch_meta = {0: exp_detail_1, 1: exp_detail_1}
master_sch_df = _priority_scheduling_df([tracker_schs.tx_schedule, fence_schs.tx_schedule], master_sch_meta)
master_sch_df

,exp_num,pointing_az,pointing_el,end_time,allowed_start_time,allowed_end_time
stt_tstmp_us,,,,,,
2025-01-01 00:00:00.000,0,-0.153980,-0.566812,2025-01-01 00:00:00.010,2025-01-01 00:00:00.010,2025-01-01 00:01:00.000
2025-01-01 00:01:00.000,0,-0.120800,-0.611894,2025-01-01 00:01:00.010,2025-01-01 00:00:00.010,2025-01-01 00:02:00.000
2025-01-01 00:02:00.000,0,-0.087521,-0.654860,2025-01-01 00:02:00.010,2025-01-01 00:01:00.010,2025-01-01 00:03:00.000
2025-01-01 00:03:00.000,0,-0.054273,-0.695517,2025-01-01 00:03:00.010,2025-01-01 00:02:00.010,2025-01-01 00:04:00.000
2025-01-01 00:04:00.000,0,-0.021183,-0.733684,2025-01-01 00:04:00.010,2025-01-01 00:03:00.010,2025-01-01 00:05:00.000
...,...,...,...,...,...,...
2025-01-01 03:59:59.950,1,270.000000,42.307692,2025-01-01 03:59:59.960,2025-01-01 03:59:59.950,2025-01-01 03:59:59.960
2025-01-01 03:59:59.960,1,270.000000,39.230769,2025-01-01 03:59:59.970,2025-01-01 03:59:59.960,2025-01-01 03:59:59.970
2025-01-01 03:59:59.970,1,270.000000,36.153846,2025-01-01 03:59:59.980,2025-01-01 03:59:59.970,2025-01-01 03:59:59.980


In [ ]:
master_sch = Schedule.from_dataframe(master_sch_df, master_sch_meta)
master_sch

Schedule(stt_tstmp_us=array(['2025-01-01T00:00:00.000000', '2025-01-01T00:01:00.000000',
       '2025-01-01T00:02:00.000000', ..., '2025-01-01T03:59:59.970000',
       '2025-01-01T03:59:59.980000', '2025-01-01T03:59:59.990000'],
      shape=(6239,), dtype='datetime64[us]'), exp_num=array([0, 0, 0, ..., 1, 1, 1], shape=(6239,)), pointing_az=array([-1.53980434e-01, -1.20799536e-01, -8.75209343e-02, ...,
        2.70000000e+02,  2.70000000e+02,  2.70000000e+02], shape=(6239,)), pointing_el=array([-0.5668117 , -0.61189391, -0.6548596 , ..., 36.15384615,
       33.07692308, 30.        ], shape=(6239,)))